In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

/Users/skamateros/Desktop/SU/thesis/priv_github/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
data = pd.read_csv('data/SU.csv', sep=';')
print(data.columns)

Index(['id', 'revoked', 'department_number', 'value', 'identifier', 'name',
       'name_en', 'credits', 'value.1', 'start_date', 'education_area_list_id',
       'percentage', 'value.2', 'name.1', 'value.3', 'name.2', 'value.4',
       'Saknar huvudområde', 'Saknar huvudområde.1', 'plan_description',
       'objectives', 'form_of_instruction', 'examination', 'literature',
       'transitional_rules', 'limitations', 'additional_information',
       'decision_date', 'decision_info'],
      dtype='str')


In [11]:
for name in list(data['name.2'].unique()):
    print(name)

Fysik
Japanska
Geovetenskap
Litteraturvetenskap
Dans- och teatervetenskap
Nationalekonomi
Statistik
Juridik och rättsvetenskap
Pedagogik
Barn- och ungdomsstudier
Religionsvetenskap
Länderkunskap/länderstudier
Koreanska
Franska
Ledarskap, organisation och styrning
Geovetenskap och naturgeografi
Spanska
Folkhälsovetenskap
Utbildningsvetenskap praktisk-estetiska ämnen
Utbildningsvetenskap teoretiska ämnen
Filosofi
Utbildningsvetenskap/didaktik allmänt
Övrigt inom naturvetenskap
Biologi
Översättning och tolkning
Studie- och yrkesvägledning
Engelska
Övrigt inom samhällsvetenskap
Kemi
Matematik
Informatik/Data- och systemvetenskap
Utbildningsvetenskap
Företagsekonomi
Konstvetenskap
Data och systemvetenskap
Idé- och lärdomshistoria/Idéhistoria
Psykologi
Socialt arbete och social omsorg
Historia
Socialantropologi
Svenska som andraspråk
Svenska/Nordiska språk
Sociologi
Matematisk statistik
Samhällskunskap
Psykoterapi
Miljövetenskap
Tyska
Filmvetenskap
Arkeologi
Kriminologi
Finska
Genusstudier
N

In [ ]:
# keep most recent row per course based on start_date
selected_columns = data[['identifier','start_date','name.2','name','plan_description','objectives']]
selected_columns['start_date'] = pd.to_datetime(selected_columns['start_date'], errors='coerce')
selected_columns = (selected_columns.sort_values(['identifier', 'start_date'], ascending=[True, False])
                                  .drop_duplicates(subset='identifier', keep='first')
                                  .reset_index(drop=True))
selected_columns.to_csv('data/SU_only_most_recent.csv', sep=";", index=False)

In [ ]:
file = pd.read_csv('data/SU_only_most_recent.csv', sep=';')
file.head()

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
sentences = ["Det här är en exempelmening", "Det här är en annan exempelmening"]

model = SentenceTransformer('KBLab/sentence-bert-swedish-cased')
embeddings = model.encode(sentences)
print(embeddings)

similarity = embeddings[0] @ embeddings[1] / (np.linalg.norm(embeddings[0]) * np.linalg.norm(embeddings[1]))
print(f"Cosine similarity between the two sentences: {similarity}")

In [ ]:
plans = file['plan_description'].tolist()
plans_embed = dict(zip(plans, model.encode(plans)))

In [ ]:
query = "redogöra för medierättens komplicerade regelstruktur och systematik och dess rättskällor"
query_embed = model.encode(query)
similarities = {plan: query_embed @ embed / (np.linalg.norm(query_embed) * np.linalg.norm(embed)) for plan, embed in plans_embed.items()}
sorted_plans = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
print("Top 5 most similar plans:")
for plan, sim in sorted_plans[:5]:
    print(f"Plan: {plan}, Similarity: {sim}")

In [ ]:
class Outcome:
    def __init__(self, outcome, code):
        self.outcome = outcome
        self.code = code
        self.embedding = model.encode(outcome)
    
    def __str__(self):
        return self.outcome

In [ ]:
class Plan:
    def __init__(self, description, code):
        self.description = description
        self.code = code
        self.embedding = model.encode(description)

    def __str__(self):
        return self.description

In [4]:
model = SentenceTransformer('KBLab/sentence-bert-swedish-cased')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1681.91it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: KBLab/sentence-bert-swedish-cased
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import json 

with open('data/SU.heuristics.json', 'r') as f:
    corpus = json.load(f)

outcomes = []
plans = []

for item in corpus['Course-list']:
    plans.append(Plan(item['CourseContent']))
    for outcome in item['ILO-list-sv']:
        outcomes.append(Outcome(outcome, item['CourseCode']))

In [6]:
print(outcomes[0].outcome)
print(outcomes[0].embedding)

Du ska kunna inom ramen för Big Bang .
[ 1.51605820e-02  4.78789695e-02 -3.95076722e-02 -1.00453593e-01
  3.16731296e-02  2.80176271e-02 -2.10365597e-02  1.46805076e-02
 -7.62492418e-05  6.37527630e-02  9.41568688e-02 -1.13175221e-01
 -8.85301828e-02  2.69488413e-02  1.55653089e-01 -6.52271211e-02
  3.09697110e-02 -8.11905321e-03 -7.61107877e-02 -2.05431357e-02
 -1.03799962e-01 -6.81869909e-02 -1.26404548e-02 -8.14206302e-02
 -2.35437509e-02 -7.20495060e-02  2.38245782e-02 -2.63016596e-02
 -3.16220783e-02  5.85479960e-02 -6.47684559e-02 -4.05033678e-02
 -5.73441274e-02  1.04603678e-01  4.76718151e-06 -3.29227783e-02
 -3.38651054e-02 -1.91846993e-02 -8.77293572e-02  2.42014937e-02
 -2.84314472e-02  1.18908904e-01 -3.04661673e-02  5.39758354e-02
 -4.40877303e-02  1.11683674e-01  8.27592686e-02  3.33381034e-02
 -7.33889788e-02  8.98019969e-02 -1.49011174e-02 -7.28606284e-02
  3.08286659e-02 -1.15777232e-01  1.08369917e-01  4.55006137e-02
  7.57999439e-03 -2.00919411e-03  7.33476132e-02  1

In [7]:
print(outcomes[1].outcome)

Du ska kunna teorin kunna ge en orienterande redogörelse för universums uppkomst, struktur och utveckling fram till idag .
